In [1]:
import sys
sys.path.append('/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI')

import torch
import numpy as np

from utils import get_network, get_loader
from FaultGenerators.WeightFaultInjector import WeightFaultInjector
from BERCampaign import BERCampaign

In [2]:
ROOT    = '/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI'
DATASET_ROOT = f'{ROOT}/data'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NETWORK = 'VGG13_BN_SGD_EP_200_LR_04'
DATASET = 'CIFAR100'

network = get_network(network_name=NETWORK, device=DEVICE, dataset_name=DATASET, root=ROOT)
_, loader = get_loader(network_name=NETWORK, batch_size=256, dataset_name=DATASET, root=DATASET_ROOT)
injector = WeightFaultInjector(network)

print(f'Device: {DEVICE}')
print(f'Network loaded: {NETWORK}')

Loading network VGG13_BN_SGD_EP_200_LR_04
Loading CIFAR100 dataset
Dataset loaded
Batch size:		256 
Number of batches:	40
Device: cuda
Network loaded: VGG13_BN_SGD_EP_200_LR_04


In [3]:
total_bits = 300_865_536
p_values = [1e-8, 5e-8, 1e-7, 5e-7]
injection_levels = [max(1, round(p * total_bits)) for p in p_values]

print('Injection levels grid:')
for p, k in zip(p_values, injection_levels):
    print(f'  p={p:.0e}  →  k={k} bits')

campaign = BERCampaign(
    network=network,
    loader=loader,
    injector=injector,
    device=DEVICE,
    injection_levels=injection_levels,
    sampling_mode='constant',
    network_name=NETWORK,
    dataset_name=DATASET,
    root=ROOT,
    pilot_trials=20,
    max_trials=100,
    precision_e=0.01,
    confidence_t=2.576,
    seed=51195,
)

Injection levels grid:
  p=1e-08  →  k=3 bits
  p=5e-08  →  k=15 bits
  p=1e-07  →  k=30 bits
  p=5e-07  →  k=150 bits


In [4]:
results = campaign.run()

Computing golden outputs...
Golden outputs saved to /home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI/output/golden_output_ber/VGG13_BN_SGD_EP_200_LR_04_CIFAR100_golden.pt
Golden accuracy: 0.7138



BER Campaign:  25%|██▌       | 1/4 [03:10<09:32, 190.85s/it]

[WARNING] injection_level=3: effective half-width 0.0883 > precision_e 0.0100. Consider increasing max_trials.
[BERCampaign] level=3 | trials=100 | M=0.5208 | crit=0.0251 | std_M=0.3428 | half_width=0.0883


BER Campaign:  50%|█████     | 2/4 [06:22<06:22, 191.11s/it]

[WARNING] injection_level=15: effective half-width 0.0107 > precision_e 0.0100. Consider increasing max_trials.
[BERCampaign] level=15 | trials=100 | M=0.0173 | crit=0.1605 | std_M=0.0415 | half_width=0.0107


BER Campaign:  75%|███████▌  | 3/4 [09:34<03:11, 191.65s/it]

[BERCampaign] level=30 | trials=100 | M=0.0014 | crit=0.2879 | std_M=0.0081 | half_width=0.0021


BER Campaign: 100%|██████████| 4/4 [12:36<00:00, 189.20s/it]

[BERCampaign] level=150 | trials=92 | M=0.0000 | crit=0.9024 | std_M=0.0000 | half_width=0.0000


In [5]:
golden = list(results.values())[0]['golden_accuracy']
print(f'golden_accuracy = {golden:.4f}\n')

for level, r in results.items():
    print(f"injection_level = {level}")
    print(f"  mean masking ratio   = {r['mean_masking_ratio']:.4f}")
    print(f"  mean critical ratio  = {r['mean_critical_ratio']:.4f}")
    print(f"  mean faulty accuracy = {r['mean_faulty_accuracy']:.4f}")
    print(f"  sizing metric        = {r['sizing_metric']}")
    print(f"  n_trials             = {r['n_trials']}")
    print(f"  half_width           = {r['effective_half_width']:.4f}")
    print()

golden_accuracy = 0.7138

injection_level = 3
  mean masking ratio   = 0.5208
  mean critical ratio  = 0.0251
  mean faulty accuracy = 0.6961
  sizing metric        = masking
  n_trials             = 100
  half_width           = 0.0883

injection_level = 15
  mean masking ratio   = 0.0173
  mean critical ratio  = 0.1605
  mean faulty accuracy = 0.6003
  sizing metric        = accuracy
  n_trials             = 100
  half_width           = 0.0107

injection_level = 30
  mean masking ratio   = 0.0014
  mean critical ratio  = 0.2879
  mean faulty accuracy = 0.5106
  sizing metric        = accuracy
  n_trials             = 100
  half_width           = 0.0021

injection_level = 150
  mean masking ratio   = 0.0000
  mean critical ratio  = 0.9024
  mean faulty accuracy = 0.0731
  sizing metric        = accuracy
  n_trials             = 92
  half_width           = 0.0000

